# Garment Worker Productivity Prediction

> **Academic group project** completed as part of a college academic internship. This repository documents collaborative coursework; individual contributions should be credited separately.

## Before running
1. Obtain the project dataset (`garments_worker_productivity.csv`) from the original authorised course/internship source.
2. Place the CSV in the same directory as this notebook.
3. Install the dependencies listed in `requirements.txt`.
4. Run the notebook from top to bottom.

The dataset is intentionally not bundled here because its redistribution permissions have not been verified. Do not publish restricted or personally identifiable data.



In [ ]:
import numpy as np
import pandas as pd

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
file_path = "garments_worker_productivity.csv"

In [ ]:
gdf = pd.read_csv(file_path)

In [ ]:
gdf.shape

In [ ]:
gdf.head()

In [ ]:
gdf.info()

In [ ]:
string_cols = gdf.select_dtypes(include=['object']).columns
gdf[string_cols] = gdf[string_cols].apply(lambda x: x.str.strip())

In [ ]:
gdf['department'] = gdf['department'].replace({'sweing': 'sewing'})

In [ ]:
gdf.isnull().sum()[gdf.isnull().sum() > 0]

In [ ]:
gdf['wip_missing_flag'] = gdf['wip'].isnull().astype(int)

In [ ]:
categorical_cols = ['department', 'quarter', 'day']

In [ ]:
for col in categorical_cols:
  print(f"\nMissing WIP by {col.upper()}")
  print(gdf.groupby(col)['wip_missing_flag'].agg(
  Total_Records='count',
  Missing_Count='sum',
  Pct_Missing=lambda x: (x.sum() / x.count()) * 100
  ).round(2))

In [ ]:
target_impact = gdf.groupby('wip_missing_flag')['actual_productivity'].mean()
print(target_impact)

Handling Missing Data: WIP

1. Mechanism: Missing Not At Random (MNAR).
2. Business Logic: 100% of missing wip values originate from the Finishing department. Since this phase of production only processes completed garments, they do not carry "work in progress" inventory.
3. Imputation Action: Replaced all missing values with 0 to accurately reflect the physical operational baseline.
















In [ ]:
gdf['wip'] = gdf['wip'].fillna(0)

In [ ]:
gdf.isnull().sum()

In [ ]:
gdf.duplicated().sum()

In [ ]:
weekend_days = ['Saturday', 'Sunday']
gdf['is_weekend'] = gdf['day'].isin(weekend_days).astype(int)

In [ ]:
eda_df=gdf.copy()

In [ ]:
eda_df.drop(columns=['date',  'wip_missing_flag'], inplace=True)

In [ ]:
gdf1 = gdf.copy()

In [ ]:
columns_to_drop = ['date', 'day', 'wip_missing_flag']
gdf1 = gdf1.drop(columns=columns_to_drop, errors='ignore')

In [ ]:
gdf1 = pd.get_dummies(gdf1, columns=['quarter', 'department','team'], drop_first=True, dtype=int)

In [ ]:
gdf1.shape

In [ ]:
gdf1.head().transpose()

In [ ]:
eda_df.info()

In [ ]:
eda_df['team'] = eda_df['team'].astype(str)

In [ ]:
eda_df['is_weekend'] = eda_df['is_weekend'].map({0: 'Weekday', 1: 'Weekend'})

In [ ]:
cat_cols = eda_df.select_dtypes(include=["object"]).columns.tolist()

In [ ]:
print(f"Text Columns ({len(cat_cols)}):", cat_cols)

In [ ]:
for col in cat_cols:
    print(f"\nFeature: '{col}' ({eda_df[col].nunique()} unique categories)")
    freq_df = pd.DataFrame(
        {
            "Count": eda_df[col].value_counts(),
            "Percentage (%)": (eda_df[col].value_counts(normalize=True) * 100).round(2)
        }
    )
    print(freq_df)

In [ ]:
fig, axes = plt.subplots(1, len(cat_cols), figsize=(18, 5))

for idx, col in enumerate(cat_cols):
  sns.countplot(
      data=eda_df,
      y=col,
      ax=axes[idx],
      order=eda_df[col].value_counts().index,
      palette='viridis',
      hue=col,
      legend=False,
  )
  axes[idx].set_title(f'Count of {col} ', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
numeric_cols = eda_df.select_dtypes(include=['int64', 'float64']).columns.tolist()

In [ ]:
print(f"Numeric Columns ({len(numeric_cols)}):", numeric_cols)

In [ ]:
num_summary = pd.concat([
    eda_df[numeric_cols].describe().round(2).T,
    eda_df[numeric_cols].skew().round(2).rename('skewness'),
    eda_df[numeric_cols].kurtosis().round(2).rename('kurtosis')
], axis=1)
display(num_summary)

In [ ]:
discrete_cols = ['no_of_workers', 'wip', 'idle_men', 'no_of_style_change']
cont_cols = ['actual_productivity', 'targeted_productivity', 'smv', 'over_time', 'incentive']

In [ ]:
plt.figure(figsize=(15, 8))
for i, col in enumerate(cont_cols, 1):
    plt.subplot(2, 3, i)
    sns.histplot(eda_df[col], kde=True, bins=30, color='royalblue')
    plt.title(f'Continuous: {col}')
    plt.xlabel(col)
    plt.ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(cont_cols), figsize=(18, 5))
axes = axes.flatten()

for i, col in enumerate(cont_cols):
    sns.boxplot(y=eda_df[col], ax=axes[i], color='royalblue')
    axes[i].set_title(f'Box Plot: {col}')
    axes[i].set_ylabel(col)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

# 1. no_of_workers: Use histplot with discrete=True (handles spread cleanly)
sns.histplot(eda_df['no_of_workers'].dropna(), ax=axes[0], color='royalblue', discrete=True)
axes[0].set_title('Distribution: no_of_workers')
axes[0].set_xlabel('no_of_workers')

# 2. wip: Use standard histplot with KDE since it has a wide continuous range
sns.histplot(eda_df['wip'][eda_df['wip'] > 0], ax=axes[1], color='teal', kde=False)
axes[1].set_title('Distribution: wip (Non-Zero)')
axes[1].set_xlabel('wip')

# 3. idle_men: Use countplot since it's heavily zero-inflated with very few unique values
idle_nz = eda_df['idle_men'][eda_df['idle_men'] > 0].value_counts().sort_index()
sns.barplot(x=idle_nz.index, y=idle_nz.values, ax=axes[2], color='darkorange')
axes[2].set_title('Frequency: Non-Zero Idle Men')
axes[2].set_xlabel('idle_men')
axes[2].set_ylabel('Frequency')

# 4. no_of_style_change: Use countplot (low cardinality: 0, 1, 2)
sns.countplot(x=eda_df['no_of_style_change'], ax=axes[3], palette='Set2')
axes[3].set_title('Count Plot: no_of_style_change')
axes[3].set_xlabel('no_of_style_change')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 1. Box plot for wip
sns.boxplot(y=eda_df['wip'], ax=axes[0], color='teal')
axes[0].set_title('Box Plot: wip')
axes[0].set_ylabel('wip')

# 2. Box plot for no_of_workers
sns.boxplot(y=eda_df['no_of_workers'], ax=axes[1], color='royalblue')
axes[1].set_title('Box Plot: no_of_workers')
axes[1].set_ylabel('no_of_workers')

plt.tight_layout()
plt.show()

In [ ]:
corr_matrix = eda_df[numeric_cols].corr()
print("--- Correlation Matrix ---")
print(corr_matrix[['actual_productivity']])

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5, cbar=True)
plt.title('Correlation Heatmap of Numerical Features')
plt.tight_layout()
plt.show()

Interpretation: targeted_productivity is the primary positive driver (r = 0.42). Operational disruptions (no_of_style_change at -0.21 and idle_men at -0.18) are the strongest negative linear detractors.              
Action: Note these baseline relationships, keeping in mind that high collinearity between smv and no_of_workers will need to be addressed during modeling.

In [ ]:
continuous_features = ['targeted_productivity', 'smv', 'over_time', 'incentive']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(continuous_features):
    sns.scatterplot(x=eda_df[col], y=eda_df['actual_productivity'], ax=axes[i], alpha=0.4, color='royalblue')
    sns.regplot(x=eda_df[col], y=eda_df['actual_productivity'], ax=axes[i], scatter=False, color='red')
    axes[i].set_title(f'{col} vs actual_productivity')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('actual_productivity')

plt.tight_layout()
plt.show()

Interpretation: targeted_productivity shows a clear positive linear trend. smv, over_time, and incentive show flat or weak linear trends due to heavy clustering and zero-inflation.

Action: Use non-linear or tree-based models to capture the true impact of zero-inflated features like incentives and overtime.

In [ ]:
X = eda_df.select_dtypes(include=['int64', 'float64']).drop(columns=['actual_productivity'], errors='ignore').dropna()

vif_data = pd.DataFrame()
vif_data["Feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif_data = vif_data.sort_values(by="VIF", ascending=False).reset_index(drop=True)

print("--- Variance Inflation Factor (VIF) Values ---")
print(vif_data)

Interpretation: Severe multicollinearity in no_of_workers (26.52) and smv (17.33). Moderate overlap in over_time (6.85).

Action: Drop no_of_workers for linear regression, or use tree-based models which are immune to VIF inflation.

In [ ]:
target = 'actual_productivity'

In [ ]:
for col in categorical_cols:
    summary = eda_df.groupby(col)['actual_productivity'].agg(['count', 'mean', 'median', 'min', 'max'])
    print(f" Productivity Summary by {col.upper()} ")
    print(summary)

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 15))
axes = axes.flatten()

for idx, col in enumerate(cat_cols):
    sns.barplot(
        data=eda_df,
        x=col,
        y=target,
        ax=axes[idx],
        palette='viridis',
        ci=None
    )
    axes[idx].set_title(f'Mean {target} by {col}')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel(f'Mean {target}')
    axes[idx].tick_params(axis='x', rotation=45)

# Hide any unused subplots
for idx in range(len(cat_cols), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.show()

Interpretation:                                                                  
1.Department: Finishing lines outperform sewing (medians: 0.81 vs. 0.75).          
2.Quarter: Productivity remains stable in Quarters 1–4 (medians ~0.72–0.80) with a higher median in Quarter 5 (0.89, though with a smaller sample size n=44).                                                     
3.Day: Output is uniform across all days of the week (medians ~0.75–0.80).       
4.Team: Significant performance variance exists between teams (e.g., Team 1 median is 0.85 vs. Team 7 at 0.68).

Action: Retain department, quarter, and team as categorical predictors for modeling; consider treating day as lower priority since day-of-week impact is minimal.

In [ ]:
for col in discrete_cols:
    summary = eda_df.groupby(col)['actual_productivity'].agg(['count', 'mean', 'median', 'min', 'max'])
    print(f"--- Productivity Summary by {col.upper()} ---")
    print(summary)
    print("\n" )

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, col in enumerate(discrete_cols):
    sns.lineplot(
        data=eda_df,
        x=col,
        y=target,
        ax=axes[idx],
        marker='o',
        errorbar=None,
        color='teal'
    )
    axes[idx].set_title(f'Mean {target} by {col}')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel(f'Mean {target}')

plt.tight_layout()
plt.show()

In [ ]:
# idle_time is 98.5% zero — whether idle time happened at all is the real signal.
gdf1['has_idle_time'] = (gdf1['idle_time'] > 0).astype(int)

In [ ]:
# over_time is right-skewed; log-compresses outliers.
gdf1['over_time_log'] = np.log1p(gdf1['over_time'])

In [ ]:
# Normalizes workload by team size instead of raw scale
gdf1['ot_per_worker'] = gdf1['over_time'] / (gdf1['no_of_workers'] + 1)
gdf1['wip_per_worker'] = gdf1['wip'] / (gdf1['no_of_workers'] + 1)

In [ ]:
#raw over_time (replaced by log version) + no_of_workers (severe VIF)
gdf1 = gdf1.drop(columns=['over_time', 'no_of_workers','wip'])

In [ ]:
# Export the processed dataset to the project directory.
from pathlib import Path

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)
gdf1.to_csv(output_dir / "fully_processed_garments_data.csv", index=False)
print("Saved processed dataset to outputs/fully_processed_garments_data.csv")

In [ ]:
target = 'actual_productivity'
X = gdf1.drop(columns=[target])
y = gdf1[target]

In [ ]:
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
test_sizes = [0.2, 0.25, 0.3, 0.35, 0.4]

1. Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression

for ts in test_sizes:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=ts, random_state=42)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = LinearRegression()
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    print(f"~ Linear Regression | Test size = {ts} ~")
    print("R2 Score:", r2_score(y_test, y_pred))
    print("MAE:", mean_absolute_error(y_test, y_pred))
    print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
    print()

2. KNN

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

for ts in test_sizes:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=ts, random_state=42)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = KNeighborsRegressor()
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    print(f"~ KNN | Test size = {ts} ~")
    print("R2 Score:", r2_score(y_test, y_pred))
    print("MAE:", mean_absolute_error(y_test, y_pred))
    print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
    print()

3. SVM

In [ ]:
from sklearn.svm import SVR

for ts in test_sizes:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=ts, random_state=42)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = SVR()
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    print(f"~ SVR | Test size = {ts} ~")
    print("R2 Score:", r2_score(y_test, y_pred))
    print("MAE:", mean_absolute_error(y_test, y_pred))
    print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
    print()

4. Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeRegressor

for ts in test_sizes:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=ts, random_state=42)
    model = DecisionTreeRegressor(random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print(f"~ Decision Tree | Test size = {ts} ~")
    print("R2 Score:", r2_score(y_test, y_pred))
    print("MAE:", mean_absolute_error(y_test, y_pred))
    print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
    print()

5. Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

for ts in test_sizes:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=ts, random_state=42)
    model = RandomForestRegressor(random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print(f"~ Random Forest | Test size = {ts} ~")
    print("R2 Score:", r2_score(y_test, y_pred))
    print("MAE:", mean_absolute_error(y_test, y_pred))
    print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
    print()

6. XGBoost

In [ ]:
from xgboost import XGBRegressor

for ts in test_sizes:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=ts, random_state=42)
    model = XGBRegressor(random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print(f"~ XGBoost | Test size = {ts} ~")
    print("R2 Score:", r2_score(y_test, y_pred))
    print("MAE:", mean_absolute_error(y_test, y_pred))
    print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
    print()

# CROSS VALIDATION

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Linear Regerssion CV
model = LinearRegression()
cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='r2')
print("Linear Regression CV R2:", cv_scores.mean(), "+/-", cv_scores.std())

In [ ]:
# KNN CV
model = KNeighborsRegressor()
cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='r2')
print("KNN CV R2:", cv_scores.mean(), "+/-", cv_scores.std())

In [ ]:
# SVM CV
model = SVR()
cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='r2')
print("SVR CV R2:", cv_scores.mean(), "+/-", cv_scores.std())

In [ ]:
# Decision Tree CV
model = DecisionTreeRegressor(random_state=42)
cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='r2')
print("Decision Tree CV R2:", cv_scores.mean(), "+/-", cv_scores.std())

In [ ]:
# Random Forest CV
model = RandomForestRegressor(random_state=42)
cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='r2')
print("Random Forest CV R2:", cv_scores.mean(), "+/-", cv_scores.std())

In [ ]:
# XGBoost CV
model = XGBRegressor(random_state=42)
cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='r2')
print("XGBoost CV R2:", cv_scores.mean(), "+/-", cv_scores.std())

# Fine-Tuning

In [ ]:
param_grid_svr = {"C": [0.1, 1, 10, 100], "kernel": ["linear", "rbf"], "gamma": ["scale", "auto"], "epsilon": [0.01, 0.1, 0.2]}
grid_svr = GridSearchCV(SVR(), param_grid_svr, cv=cv, scoring='r2', n_jobs=-1)
grid_svr.fit(X_train_scaled, y_train)
print("Best params:", grid_svr.best_params_)
print("Best CV R2:", grid_svr.best_score_)
best_svr = grid_svr.best_estimator_
y_pred = best_svr.predict(X_test_scaled)
print("Test R2:", r2_score(y_test, y_pred))
print("Test MAE:", mean_absolute_error(y_test, y_pred))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

In [ ]:
param_grid_knn = {
    "n_neighbors": [3, 5, 7, 9, 11, 15],
    "weights": ["uniform", "distance"],
    "p": [1, 2]
}

grid_knn = GridSearchCV(KNeighborsRegressor(), param_grid_knn, cv=cv, scoring='r2', n_jobs=-1)
grid_knn.fit(X_train_scaled, y_train)
print("Best params:", grid_knn.best_params_)
print("Best CV R2:", grid_knn.best_score_)

best_knn = grid_knn.best_estimator_
y_pred = best_knn.predict(X_test_scaled)
print("Test R2:", r2_score(y_test, y_pred))
print("Test MAE:", mean_absolute_error(y_test, y_pred))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

In [ ]:
param_grid_dt = {"max_depth": [None, 3, 5, 7, 10], "min_samples_split": [2, 5, 10], "min_samples_leaf": [1, 2, 4]}
grid_dt = GridSearchCV(DecisionTreeRegressor(random_state=42), param_grid_dt, cv=cv, scoring='r2', n_jobs=-1)
grid_dt.fit(X_train, y_train)
print("Best params:", grid_dt.best_params_)
print("Best CV R2:", grid_dt.best_score_)
best_dt = grid_dt.best_estimator_
y_pred = best_dt.predict(X_test)
print("Test R2:", r2_score(y_test, y_pred))
print("Test MAE:", mean_absolute_error(y_test, y_pred))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

In [ ]:
param_grid_rf = {"n_estimators": [100, 200, 300], "max_depth": [None, 5, 10, 15], "min_samples_split": [2, 5, 10], "min_samples_leaf": [1, 2, 4]}
grid_rf = GridSearchCV(RandomForestRegressor(random_state=42), param_grid_rf, cv=cv, scoring='r2', n_jobs=-1)
grid_rf.fit(X_train, y_train)
print("Best params:", grid_rf.best_params_)
print("Best CV R2:", grid_rf.best_score_)
best_rf = grid_rf.best_estimator_
y_pred = best_rf.predict(X_test)
print("Test R2:", r2_score(y_test, y_pred))
print("Test MAE:", mean_absolute_error(y_test, y_pred))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

In [ ]:
param_grid_xgb = {"n_estimators": [100, 200, 300], "max_depth": [3, 5, 7], "learning_rate": [0.01, 0.05, 0.1, 0.2], "subsample": [0.8, 1.0]}
grid_xgb = GridSearchCV(XGBRegressor(random_state=42), param_grid_xgb, cv=cv, scoring='r2', n_jobs=-1)
grid_xgb.fit(X_train, y_train)
print("Best params:", grid_xgb.best_params_)
print("Best CV R2:", grid_xgb.best_score_)
best_xgb = grid_xgb.best_estimator_
y_pred = best_xgb.predict(X_test)
print("Test R2:", r2_score(y_test, y_pred))
print("Test MAE:", mean_absolute_error(y_test, y_pred))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

# **ANN**

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from itertools import product

In [ ]:
histories_reg = {}
test_size_results_reg = {}

for ts in test_sizes:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=ts, random_state=42)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = Sequential()
    model.add(Dense(units=16, activation='relu', input_dim=X_train_scaled.shape[1]))
    model.add(Dropout(0.2))
    model.add(Dense(units=8, activation='relu'))
    model.add(Dense(units=1, activation='linear'))
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])

    history = model.fit(X_train_scaled, y_train, epochs=50, batch_size=32,
                         validation_split=0.2, verbose=0)

    y_pred = model.predict(X_test_scaled, verbose=0).ravel()

    histories_reg[ts] = history
    test_size_results_reg[ts] = {
        "r2": r2_score(y_test, y_pred),
        "mae": mean_absolute_error(y_test, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_test, y_pred))
    }

    print(f"~ ANN (TensorFlow) | Test size = {ts} ~")
    print("R2 Score:", test_size_results_reg[ts]["r2"])
    print("MAE:", test_size_results_reg[ts]["mae"])
    print("RMSE:", test_size_results_reg[ts]["rmse"])
    print()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, ts in enumerate(test_sizes):
    axes[i].plot(histories_reg[ts].history['loss'], label='Train Loss')
    axes[i].plot(histories_reg[ts].history['val_loss'], label='Validation Loss')
    axes[i].set_title(f"Test size = {ts}")
    axes[i].set_xlabel("Epoch")
    axes[i].set_ylabel("Loss (MSE)")
    axes[i].legend()

axes[-1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(test_sizes, [test_size_results_reg[ts]["r2"] for ts in test_sizes], marker='o', color='green')
plt.xlabel("Test Size")
plt.ylabel("R² Score")
plt.title("ANN R² Across Test Sizes")
plt.show()

In [ ]:
param_grid_ann_reg = {
    "units_1": [16, 32, 64],
    "units_2": [8, 16, 32],
    "dropout": [0.2, 0.3],
}

keys = list(param_grid_ann_reg.keys())
combinations = list(product(*param_grid_ann_reg.values()))

best_score = -np.inf
best_params = None

for combo in combinations:
    params = dict(zip(keys, combo))
    fold_scores = []

    for train_idx, val_idx in cv.split(X_train_arr):
        X_tr, X_val = X_train_arr[train_idx], X_train_arr[val_idx]
        y_tr, y_val = y_train_arr[train_idx], y_train_arr[val_idx]

        model = Sequential()
        model.add(Dense(units=params["units_1"], activation='relu', input_dim=X_tr.shape[1]))
        model.add(Dropout(params["dropout"]))
        model.add(Dense(units=params["units_2"], activation='relu'))
        model.add(Dense(units=1, activation='linear'))
        model.compile(optimizer='adam', loss='mse', metrics=['mae'])
        model.fit(X_tr, y_tr, epochs=50, batch_size=32, verbose=0)

        y_val_pred = model.predict(X_val, verbose=0).ravel()
        fold_scores.append(r2_score(y_val, y_val_pred))

    mean_score = np.mean(fold_scores)
    print(f"Params: {params} -> CV R2: {mean_score:.4f}")

    if mean_score > best_score:
        best_score = mean_score
        best_params = params

print("\nBest params:", best_params)
print("Best CV R2:", best_score)

best_model = Sequential()
best_model.add(Dense(units=best_params["units_1"], activation='relu', input_dim=X_train_arr.shape[1]))
best_model.add(Dropout(best_params["dropout"]))
best_model.add(Dense(units=best_params["units_2"], activation='relu'))
best_model.add(Dense(units=1, activation='linear'))
best_model.compile(optimizer='adam', loss='mse', metrics=['mae'])
best_model.fit(X_train_arr, y_train_arr, epochs=50, batch_size=32, verbose=0)

y_pred = best_model.predict(X_test_scaled, verbose=0).ravel()
print("Test R2:", r2_score(y_test, y_pred))
print("Test MAE:", mean_absolute_error(y_test, y_pred))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

In [ ]:
plt.figure(figsize=(7,6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel("Actual Productivity")
plt.ylabel("Predicted Productivity")
plt.title("ANN: Actual vs Predicted (Tuned)")
plt.show()